# Cadrage du Problème

L'objectif est un **classification binaire** qui indique si un vol sera en retard ou non. Retard = ArrDelay > 15 minutes

In [ ]:
# Importation des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Métriques
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score


# Chargement des données

In [ ]:
file_path = 'flight-delay-dataset-20182022/Combined_Flights_2021.parquet'

df = pd.read_parquet(file_path)

df = df.sample(n=1000000, random_state=42).reset_index(drop=True)

print(f"Dataset complet chargé : {len(df):,} vols")

## Exploration des données (EDA)


In [ ]:
# Échantillon pour exploration initiale rapide (développement uniquement)
df_sample = df.sample(n=100000, random_state=42).reset_index(drop=True)
df_sample.head()

In [ ]:
df_sample.describe()

In [ ]:
df_sample.info()

On observe des données manquantes dans plusieurs colonnes, principalement liées aux informations de vol effectives (retards, temps de taxi, etc.), ce qui est cohérent avec la présence de vols annulés ou détournés :

- **Départ** : `DepTime`, `DepDelay`, `DepDel15`, etc. (~1,8% de manquants).
- **Arrivée** : `ArrTime`, `ArrDelay`, `ArrDel15`, etc. (~2,1% de manquants).
- **Détails techniques** : `Tail_Number` (384 manquants) et `TaxiOut`/`TaxiIn`.
- **Informations de vol** : `AirTime` et `ActualElapsedTime`.

Ces valeurs devront être traitées (imputation ou suppression) avant l'entraînement du modèle de classification.

Notre objectif étant de faire une classification binaire sur `ArrDel15`, nous devons supprimer les lignes où cette variable est manquante (environ 2,1% des données). Il s'agit des vols annulés.

Les valeurs de temps (ex. `DepTime`, `ArrTime`) sont au format float (ex. 1345.0 pour 13h45), ce qui peut nécessiter une conversion en format horaire standard pour une meilleure interprétation.

`DayOfWeek` est codé de 1 (lundi) à 7 (dimanche), ce qui est utile pour capturer les variations hebdomadaires des retards.
`DayofMonth` et `Month` sont également présents, permettant d'analyser les tendances saisonnières. Ils sont codés avec des entiers.

D'après l'exploration des données, nous pouvons classifier les variables ainsi :

**Variables Catégorielles :**
*   **Identifiants & Codes :** `Airline`, `Origin`, `Dest`, `Marketing_Airline_Network`, `Operating_Airline`, `Tail_Number`, `IATA_Code_Marketing_Airline`, etc.
*   **Temporelles (discrètes) :** `Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`.
*   **Indicateurs binaires :** `Cancelled`, `Diverted`, `DepDel15`, `ArrDel15` (Target).
*   **Groupements :** `DepartureDelayGroups`, `ArrivalDelayGroups`, `DistanceGroup`, `DepTimeBlk`, `ArrTimeBlk`.

**Variables Continues (Numériques) :**
*   **Temps de vol & Retards :** `DepDelay`, `DepDelayMinutes`, `ArrDelay`, `ArrDelayMinutes`, `AirTime`, `ActualElapsedTime`, `CRSElapsedTime`, `TaxiIn`, `TaxiOut`.
*   **Distance :** `Distance`.
*   **Horaires (à convertir) :** `DepTime`, `ArrTime`, `CRSDepTime`, `CRSArrTime`, `WheelsOff`, `WheelsOn`.

**Note :** Certaines variables comme `OriginAirportID` ou `OriginStateFips` sont stockées comme des entiers (`int64`) mais sont conceptuellement des variables **catégorielles** (identifiants).


### Nettoyage des données

In [ ]:
df_clean = df[(df['Cancelled'] == False) & (df['Diverted'] == False)].copy()

# Supprimer les lignes avec ArrDel15 manquant (vols sans info de retard)
df_clean = df_clean.dropna(subset=['ArrDel15'])

print(f"Dataset nettoyé : {len(df_clean):,} vols")

### Feature Engineering

In [ ]:
df_clean['DepHour'] = df_clean['CRSDepTime'] // 100
df_clean['IsWeekend'] = (df_clean['DayOfWeek'] >= 6).astype(int)
df_clean['IsHolidayMonth'] = df_clean['Month'].isin([6, 7, 12]).astype(int)

### Séparation Train/Test - Protection contre le Data Snooping

Toutes les analyses EDA suivantes doivent être faites sur `df_train` uniquement.

In [ ]:
from sklearn.model_selection import train_test_split

# Créer le jeu de test (20%) et le mettre de côté
df_train, df_test = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['ArrDel15']  # Préserve la proportion des classes
)

print(f"Training set: {len(df_train)} samples")
print(f"Test set: {len(df_test)} samples")
print(f"\nTest set - Distribution ArrDel15:\n{df_test['ArrDel15'].value_counts(normalize=True)}")

# ⚠️ NE PLUS TOUCHER df_test jusqu'à l'évaluation finale

## Préparation des données pour le ML (Preprocessing)

In [ ]:
# Définir les features et la target
features = [
    # Temporelles
    'Month', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth',
    # Horaire
    'CRSDepTime', 'DepHour',
    # Catégorielles
    'Marketing_Airline_Network', 'Airline', 'Origin', 'Dest',
    # Distance
    'Distance'
]
target = 'ArrDel15'

# Séparation X/y pour train et test
X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_test = df_test[features].copy()
y_test = df_test[target].copy()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\n✅ {len(features)} features sélectionnées")

In [ ]:
# Identification des colonnes numériques et catégorielles
categorical_features = ['Marketing_Airline_Network', 'Airline', 'Origin', 'Dest']
numerical_features = ['Month', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth', 
                      'CRSDepTime', 'DepHour', 'Distance']

# Pipeline pour les features numériques : imputation + standardisation
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline pour les features catégorielles : imputation + encodage one-hot
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer pour appliquer les pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print(f"\nFeatures numériques ({len(numerical_features)}): {numerical_features}")
print(f"Features catégorielles ({len(categorical_features)}): {categorical_features}")
print(f"Total features: {len(numerical_features) + len(categorical_features)}")

### Exploration des données (EDA) - Suite

In [ ]:
print("Distribution de la variable cible (ArrDel15) :")
print(df_train[target].value_counts(normalize=True))
plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df_train, hue=target, palette='viridis', legend=False)
plt.title("Répartition des retards (0 = À l'heure, 1 = Retard > 15min)")
plt.show()

In [ ]:
airline_delay = df_train.groupby('Airline')[target].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(y=airline_delay.index, x=airline_delay.values, hue=airline_delay.index, palette='coolwarm', legend=False)
plt.title('Taux de retard moyen par Compagnie Aérienne')
plt.xlabel('Proportion de retards')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=target, y='Distance', data=df_train, hue=target, palette=['green', 'red'], legend=False)
plt.xticks([0, 1], ['À l\'heure', 'En retard'])
plt.title('Distribution de la distance par statut de retard')
plt.xlabel('Statut du vol')
plt.ylabel('Distance (miles)')
plt.show()

In [ ]:
# Matrice de corrélation étendue avec features temporelles et opérationnelles
corr_features = [
    # Temporel
    'Month', 'Quarter', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth',
    # Horaire
    'CRSDepTime', 'DepHour',
    # Distance/Durée
    'Distance', 'CRSElapsedTime', 'DistanceGroup',
    # Target
    target
]

corr_data = df_train[corr_features].dropna()

plt.figure(figsize=(14, 10))
correlation_matrix = corr_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            linewidths=0.5, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation étendue - Features temporelles et opérationnelles', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Affichage des corrélations avec la target triées
print("\nCorrélations avec ArrDel15 (triées par importance) :")
target_corr = correlation_matrix[target].drop(target).sort_values(ascending=False)
print(target_corr)

In [ ]:
# Relation entre distance et retard
plt.figure(figsize=(10, 6))
distance_delay = df_train.groupby('DistanceGroup')[target].mean().sort_index()
sns.lineplot(x=distance_delay.index, y=distance_delay.values, marker='o', color='coral', linewidth=2)
plt.title('Taux de retard en fonction du groupe de distance')
plt.xlabel('Groupe de distance')
plt.ylabel('Proportion de retards')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Distribution des retards par plage horaire de départ
plt.figure(figsize=(12, 6))
delay_by_time = df_train.groupby('DepTimeBlk')[target].mean().sort_values(ascending=False)
sns.barplot(x=delay_by_time.index, y=delay_by_time.values, hue=delay_by_time.index, palette='viridis', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Taux de retard par plage horaire de départ')
plt.xlabel('Plage horaire')
plt.ylabel('Proportion de retards')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des retards par mois
plt.figure(figsize=(10, 5))
month_labels = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
delay_by_month = df_train.groupby('Month')[target].mean().sort_index()
sns.barplot(x=delay_by_month.index, y=delay_by_month.values, palette='Oranges_d', hue=delay_by_month.index, legend=False)
plt.xticks(range(12), month_labels)
plt.title('Taux de retard par mois')
plt.xlabel('Mois')
plt.ylabel('Proportion de retards')
plt.show()

# 4. Sélection et Entraînement des Modèles

Nous allons tester plusieurs modèles de classification en suivant une approche progressive :
1. **Logistic Regression** : Modèle linéaire simple (baseline)
2. **Random Forest** : Modèle d'ensemble basé sur des arbres de décision
3. Comparaison et sélection du meilleur modèle

## 4.1 Baseline : Logistic Regression

Commençons par un modèle linéaire simple pour établir une baseline de performance.

In [ ]:
from sklearn.model_selection import cross_val_score
import time

# Créer un pipeline complet : preprocessing + modèle
logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])

print("🔄 Entraînement de la Logistic Regression...")
print(f"Dataset : {len(X_train):,} échantillons d'entraînement\n")

start_time = time.time()

# Entraînement sur le training set
logistic_pipeline.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"✅ Entraînement terminé en {training_time:.2f} secondes")
print(f"Temps par échantillon : {training_time/len(X_train)*1000:.4f} ms")

In [ ]:
# Prédictions sur le training set
y_train_pred_lr = logistic_pipeline.predict(X_train)

# Métriques sur le training set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

train_accuracy_lr = accuracy_score(y_train, y_train_pred_lr)
train_precision_lr = precision_score(y_train, y_train_pred_lr)
train_recall_lr = recall_score(y_train, y_train_pred_lr)
train_f1_lr = f1_score(y_train, y_train_pred_lr)
train_roc_auc_lr = roc_auc_score(y_train, logistic_pipeline.predict_proba(X_train)[:, 1])

print("📊 Performance sur le Training Set :")
print(f"{'='*50}")
print(f"Accuracy  : {train_accuracy_lr:.4f}")
print(f"Precision : {train_precision_lr:.4f} (parmi les prédictions 'retard', combien sont correctes)")
print(f"Recall    : {train_recall_lr:.4f} (parmi les vrais retards, combien sont détectés)")
print(f"F1-Score  : {train_f1_lr:.4f} (moyenne harmonique précision/recall)")
print(f"ROC-AUC   : {train_roc_auc_lr:.4f} (capacité à distinguer les classes)")
print(f"{'='*50}")

## 4.2 Random Forest Classifier

Testons maintenant un modèle d'ensemble plus puissant basé sur des arbres de décision.

In [ ]:
# Créer un pipeline avec Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print("🌲 Entraînement du Random Forest Classifier...")
print(f"Dataset : {len(X_train):,} échantillons d'entraînement")
print(f"Hyperparamètres : 100 arbres, max_depth=20\n")

start_time = time.time()

# Entraînement
rf_pipeline.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"\n✅ Entraînement terminé en {training_time:.2f} secondes ({training_time/60:.2f} minutes)")
print(f"Temps par échantillon : {training_time/len(X_train)*1000:.4f} ms")

In [ ]:
# Prédictions sur le training set
y_train_pred_rf = rf_pipeline.predict(X_train)

# Métriques sur le training set
train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
train_precision_rf = precision_score(y_train, y_train_pred_rf)
train_recall_rf = recall_score(y_train, y_train_pred_rf)
train_f1_rf = f1_score(y_train, y_train_pred_rf)
train_roc_auc_rf = roc_auc_score(y_train, rf_pipeline.predict_proba(X_train)[:, 1])

print("📊 Performance sur le Training Set :")
print(f"{'='*50}")
print(f"Accuracy  : {train_accuracy_rf:.4f}")
print(f"Precision : {train_precision_rf:.4f}")
print(f"Recall    : {train_recall_rf:.4f}")
print(f"F1-Score  : {train_f1_rf:.4f}")
print(f"ROC-AUC   : {train_roc_auc_rf:.4f}")
print(f"{'='*50}")

## 4.3 Comparaison des Modèles

Comparons les performances des deux modèles sur le training set.

In [ ]:
# Créer un DataFrame de comparaison
comparison_df = pd.DataFrame({
    'Modèle': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [train_accuracy_lr, train_accuracy_rf],
    'Precision': [train_precision_lr, train_precision_rf],
    'Recall': [train_recall_lr, train_recall_rf],
    'F1-Score': [train_f1_lr, train_f1_rf],
    'ROC-AUC': [train_roc_auc_lr, train_roc_auc_rf]
})

print("📊 COMPARAISON DES MODÈLES (Training Set)")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Identifier le meilleur modèle
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Modèle']
print(f"\n🏆 Meilleur modèle (F1-Score) : {best_model_name}")
print(f"   F1-Score : {comparison_df.loc[best_model_idx, 'F1-Score']:.4f}")

In [ ]:
# Visualisation de la comparaison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Barplot des métriques
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, comparison_df.iloc[0, 1:].values, width, label='Logistic Regression', alpha=0.8)
axes[0].bar(x + width/2, comparison_df.iloc[1, 1:].values, width, label='Random Forest', alpha=0.8)
axes[0].set_xlabel('Métriques')
axes[0].set_ylabel('Score')
axes[0].set_title('Comparaison des Performances - Training Set')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1])

# Graphique 2 : Matrice de confusion pour le meilleur modèle
from sklearn.metrics import confusion_matrix
if best_model_name == 'Random Forest':
    cm = confusion_matrix(y_train, y_train_pred_rf)
    best_pipeline = rf_pipeline
else:
    cm = confusion_matrix(y_train, y_train_pred_lr)
    best_pipeline = logistic_pipeline

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Matrice de Confusion - {best_model_name}')
axes[1].set_xlabel('Prédiction')
axes[1].set_ylabel('Vérité')
axes[1].set_xticklabels(['À l\'heure', 'En retard'])
axes[1].set_yticklabels(['À l\'heure', 'En retard'])

plt.tight_layout()
plt.show()

## 4.4 Évaluation Finale sur le Test Set

⚠️ **IMPORTANT** : Nous évaluons maintenant le meilleur modèle sur le **test set** qui n'a **jamais été vu** pendant l'entraînement.

In [ ]:
# Prédictions sur le test set avec les deux modèles
print("🔍 Évaluation sur le Test Set...")
print(f"Test set : {len(X_test):,} échantillons\n")

# Logistic Regression
y_test_pred_lr = logistic_pipeline.predict(X_test)
test_accuracy_lr = accuracy_score(y_test, y_test_pred_lr)
test_precision_lr = precision_score(y_test, y_test_pred_lr)
test_recall_lr = recall_score(y_test, y_test_pred_lr)
test_f1_lr = f1_score(y_test, y_test_pred_lr)
test_roc_auc_lr = roc_auc_score(y_test, logistic_pipeline.predict_proba(X_test)[:, 1])

# Random Forest
y_test_pred_rf = rf_pipeline.predict(X_test)
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)
test_precision_rf = precision_score(y_test, y_test_pred_rf)
test_recall_rf = recall_score(y_test, y_test_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf)
test_roc_auc_rf = roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:, 1])

# Comparaison Test Set
test_comparison_df = pd.DataFrame({
    'Modèle': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [test_accuracy_lr, test_accuracy_rf],
    'Precision': [test_precision_lr, test_precision_rf],
    'Recall': [test_recall_lr, test_recall_rf],
    'F1-Score': [test_f1_lr, test_f1_rf],
    'ROC-AUC': [test_roc_auc_lr, test_roc_auc_rf]
})

print("📊 RÉSULTATS SUR LE TEST SET")
print("="*70)
print(test_comparison_df.to_string(index=False))
print("="*70)

# Meilleur modèle
best_test_idx = test_comparison_df['F1-Score'].idxmax()
best_test_model = test_comparison_df.loc[best_test_idx, 'Modèle']
print(f"\n🏆 Meilleur modèle (Test Set) : {best_test_model}")
print(f"   F1-Score : {test_comparison_df.loc[best_test_idx, 'F1-Score']:.4f}")

In [ ]:
# Rapport de classification détaillé pour le meilleur modèle
from sklearn.metrics import classification_report

if best_test_model == 'Random Forest':
    y_test_pred_best = y_test_pred_rf
else:
    y_test_pred_best = y_test_pred_lr

print(f"\n📋 RAPPORT DE CLASSIFICATION DÉTAILLÉ - {best_test_model}")
print("="*70)
print(classification_report(y_test, y_test_pred_best, 
                          target_names=['À l\'heure (0)', 'En retard (1)'],
                          digits=4))

# Matrice de confusion
cm_test = confusion_matrix(y_test, y_test_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['À l\'heure', 'En retard'],
            yticklabels=['À l\'heure', 'En retard'])
plt.title(f'Matrice de Confusion - Test Set\n{best_test_model}', fontsize=14, pad=20)
plt.xlabel('Prédiction', fontsize=12)
plt.ylabel('Vérité', fontsize=12)
plt.tight_layout()
plt.show()

# Analyse des erreurs
tn, fp, fn, tp = cm_test.ravel()
print(f"\n🔍 ANALYSE DES PRÉDICTIONS (Test Set) :")
print(f"{'='*70}")
print(f"Vrais Négatifs (TN)  : {tn:,} vols correctement prédits à l'heure")
print(f"Vrais Positifs (TP)  : {tp:,} vols correctement prédits en retard")
print(f"Faux Positifs (FP)   : {fp:,} vols prédits en retard mais à l'heure (fausse alarme)")
print(f"Faux Négatifs (FN)   : {fn:,} vols prédits à l'heure mais en retard (manqués)")
print(f"{'='*70}")
print(f"\nTaux de fausses alarmes : {fp/(fp+tn)*100:.2f}%")
print(f"Taux de retards manqués : {fn/(fn+tp)*100:.2f}%")

In [ ]:
from sklearn.model_selection import cross_val_score

print("🧪 Lancement de la Cross-Validation (5 folds)...")

# On utilise le ROC-AUC car c'est la métrique la plus stable
cv_scores = cross_val_score(rf_pipeline, X_train, y_train,
                            cv=5,
                            scoring='roc_auc',
                            n_jobs=-1)

print(f"Moyenne ROC-AUC : {cv_scores.mean():.4f}")
print(f"Écart-type (stabilité) : {cv_scores.std():.4f}")